In [ ]:
from ecephys_setup import get_session, get_units, get_spikes, clean_grating_spike_data

import numpy as np
import pandas as pd

import torch  
import torch.nn as nn  
import torch.optim as optim
import torch.nn.functional as F 
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

In [ ]:
visp_spikes = pd.read_csv("../audrey-data/visp_spikes.csv")
visal_spikes = pd.read_csv("../audrey-data/visal_spikes.csv")
drifting_table = pd.read_csv("../audrey-data/drifting_table.csv")

visp_spikes

In [ ]:
visp_spikes_int = visp_spikes.multiply(2).round().astype(int)
visp_spikes_int

In [ ]:
orientation_labels = drifting_table[drifting_table["orientation"].notna()]["orientation"]
orientation_labels

In [ ]:
# Use GPU if available  
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# VAE encoder
class Encoder(nn.Module):  
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):  
        super(Encoder, self).__init__()  
        self.fc1 = nn.Linear(input_dim, hidden_dim)  
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)  
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim) 

    def forward(self, X): 
        h = torch.relu(self.fc1(X)) 
        mu = self.fc_mu(h) 
        logvar = self.fc_logvar(h) 
        return mu, logvar

# VAE decoder
class Decoder(nn.Module):  
    def __init__(self, latent_dim=20, hidden_dim=400, output_dim=784, likelihood="gaussian"):  
        super(Decoder, self).__init__()
        self.likelihood = likelihood
        self.fc1 = nn.Linear(latent_dim, hidden_dim)  
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, z):
        h = torch.relu(self.fc1(z))
        out = self.fc2(h)
        if self.likelihood == "poisson":
            return F.softplus(out)      # rate λ > 0
        else:
            return out                  # Gaussian mean (real-valued)

In [ ]:
# VAE class, combining encoder and decoder
class VAE(nn.Module):  
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20, likelihood="gaussian"):  
        super(VAE, self).__init__()  
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)  
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim, likelihood=likelihood)  
    
    def reparameterize(self, mu, logvar):  
        std = torch.exp(0.5 * logvar)  # Standard deviation  
        eps = torch.randn_like(std)    # Random noise  
        return mu + eps * std          # Sample from the latent distribution  
        
    def forward(self, X):  
        mu, logvar = self.encoder(X)  
        z = self.reparameterize(mu, logvar)  
        reconstructed = self.decoder(z)
        return reconstructed, mu, logvar

# VAE loss, where total loss = reconstruction loss + KL divergence
def vae_loss(recon, X, mu, logvar, likelihood):
    # Reconstruction: negative log-likelihood, averaged over batch
    if likelihood == "gaussian":
        # Gaussian NLL ∝ MSE
        recon_loss = F.mse_loss(recon, X, reduction="mean")
    elif likelihood == "poisson":
        # Poisson NLL = mean over batch of sum over units (λ - k·log λ)
        log_factorial = torch.lgamma(X + 1)
        recon_loss = (recon - X * torch.log(recon + 1e-8) + log_factorial).sum(dim=1).mean()
        recon_loss = recon_loss / (X.mean() + 1e-8)
    # KL averaged over batch
    kl_loss = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1).mean()
    return recon_loss + kl_loss, recon_loss, kl_loss

In [ ]:
def run_VAE(X, likelihood, batch_size, n_epochs, learning_rate, latent_dim=2):
    # Initialize batches and model
    loader = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=True)
    
    input_dim = X.shape[1]
    model = VAE(input_dim=input_dim, latent_dim=latent_dim, likelihood=likelihood).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Track losses 
    train_losses = []
    recon_losses = []
    kl_losses = []
    
    # Training loop
    for epoch in range(n_epochs+1):
        model.train()
    
        total_train_loss = 0
        total_recon_loss = 0
        total_kl_loss = 0
        for (xb,) in loader:
            optimizer.zero_grad()
            recon, mu, logvar = model(xb)
            
            train_loss, recon_loss, kl_loss = vae_loss(recon, xb, mu, logvar, likelihood)
            total_train_loss += train_loss.item()
            total_recon_loss += recon_loss.item()
            total_kl_loss += kl_loss.item()
            
            train_loss.backward()
            optimizer.step()

        train_losses.append(total_train_loss)
        recon_losses.append(total_recon_loss)
        kl_losses.append(total_kl_loss)
        if epoch % 10 == 0:
            print(f"epoch {epoch}: loss={total_train_loss:.3f} recon={total_recon_loss:.3f} kl={total_kl_loss:.3f}")

    return model

In [ ]:
# Set training parameters 
n_epochs = 256
batch_size = 64
learning_rate = 1e-3 
likelihood = "gaussian"

# Initialize model, optimizer 
X = torch.tensor(visp_spikes_int.values, dtype=torch.float32).to(device)

vae_2d_model_g = run_VAE(X, likelihood, batch_size, n_epochs, learning_rate, latent_dim=2)

vae_2d_model_g.eval()
with torch.no_grad():
    mu_all, _ = vae_2d_model_g.encoder(X) # encode all available data
    Z = mu_all.cpu().numpy()
    
cmap = plt.get_cmap("tab20")   # 10 distinct colors, take first 8
orients = np.arange(0, 360, 45)
colors = [cmap(i) for i in range(len(orients))]
norm = mcolors.BoundaryNorm(boundaries=np.arange(-45 / 2, 360, 45), ncolors=8)

sc = plt.scatter(Z[:, 0], Z[:, 1], c=orientation_labels, cmap=cmap, norm=norm, s=10)
cbar = plt.colorbar(sc, ticks=orients)
cbar.ax.set_yticklabels([f"{o}°" for o in orients])
cbar.set_label("orientation")
plt.title("VAE - Gaussian Loss")
plt.show()

In [ ]:
# Set training parameters 
n_epochs = 256
batch_size = 64
learning_rate = 1e-3 
likelihood = "poisson"

# Initialize model, optimizer 
X = torch.tensor(visp_spikes_int.values, dtype=torch.float32).to(device)

vae_2d_model_p = run_VAE(X, likelihood, batch_size, n_epochs, learning_rate, latent_dim=2)

vae_2d_model_p.eval()
with torch.no_grad():
    mu_all, _ = vae_2d_model_p.encoder(X) # encode all available data
    Z = mu_all.cpu().numpy()
    
cmap = plt.get_cmap("tab20")   # 10 distinct colors, take first 8
orients = np.arange(0, 360, 45)
colors = [cmap(i) for i in range(len(orients))]
norm = mcolors.BoundaryNorm(boundaries=np.arange(-45 / 2, 360, 45), ncolors=8)

sc = plt.scatter(Z[:, 0], Z[:, 1], c=orientation_labels, cmap=cmap, norm=norm, s=10)
cbar = plt.colorbar(sc, ticks=orients)
cbar.ax.set_yticklabels([f"{o}°" for o in orients])
cbar.set_label("orientation")
plt.title("VAE - Poisson Loss")
plt.show()

## Let's do 3D, just for fun!

In [ ]:
likelihood = "gaussian"
vae_3d_model_g = run_VAE(X, likelihood, batch_size, n_epochs, learning_rate, latent_dim=3)

vae_3d_model_g.eval()
with torch.no_grad():
    mu_all, _ = vae_3d_model_g.encoder(X)
    Z = mu_all.cpu().numpy() # (n_trials, 3)

cmap = plt.get_cmap("tab20") # 10 distinct colors, take first 8
colors = [cmap(i) for i in range(8)]
norm = mcolors.BoundaryNorm(boundaries=np.arange(-45 / 2, 360, 45), ncolors=8)

# --- figure layout: 2 rows x 3 cols; 3D spans left column (both rows) ---
fig = plt.figure(figsize=(15, 9))
gs = fig.add_gridspec(2, 3, width_ratios=[1.6, 1, 1], wspace=0.25, hspace=0.3)

# ---- (a) 3D scatter (spans left column) ----
ax3d = fig.add_subplot(gs[:, 0], projection="3d")
sc = ax3d.scatter(Z[:, 0], Z[:, 1], Z[:, 2],
                  c=orientation_labels, cmap=cmap, norm=norm,
                  s=8, alpha=0.6)
ax3d.set_xlabel("z1"); ax3d.set_ylabel("z2"); ax3d.set_zlabel("z3")
ax3d.set_title(f"3D Latent — Gaussian")
ax3d.view_init(elev=20, azim=35)

# ---- (b) three pairwise 2D projections ----
pairs = [(0, 1), (0, 2), (1, 2)]
proj_axes = [
    fig.add_subplot(gs[0, 1]),  # z1 vs z2
    fig.add_subplot(gs[0, 2]),  # z1 vs z3
    fig.add_subplot(gs[1, 1]),  # z2 vs z3
]
for ax, (i, j) in zip(proj_axes, pairs):
    ax.scatter(Z[:, i], Z[:, j], c=orientation_labels,
               cmap=cmap, norm=norm, s=8, alpha=0.6)
    ax.set_xlabel(f"z{i+1}"); ax.set_ylabel(f"z{j+1}")
    ax.set_title(f"z{i+1} vs z{j+1}")
    ax.set_aspect("equal", adjustable="datalim")

# ---- (c) shared colorbar in the bottom-right empty cell ----
cax = fig.add_subplot(gs[1, 2])
cbar = fig.colorbar(sc, cax=cax, ticks=orients)
cbar.ax.set_yticklabels([f"{o}°" for o in orients])
cbar.set_label("orientation")

plt.suptitle(f"3D Latent Space — Gaussian Likelihood", fontsize=14)
plt.show()

In [ ]:
likelihood = "poisson"
vae_3d_model_p = run_VAE(X, likelihood, batch_size, n_epochs, learning_rate, latent_dim=3)

vae_3d_model_p.eval()
with torch.no_grad():
    mu_all, _ = vae_3d_model_p.encoder(X)
    Z = mu_all.cpu().numpy() # (n_trials, 3)

cmap = plt.get_cmap("tab20") # 10 distinct colors, take first 8
colors = [cmap(i) for i in range(8)]
norm = mcolors.BoundaryNorm(boundaries=np.arange(-45 / 2, 360, 45), ncolors=8)

# --- figure layout: 2 rows x 3 cols; 3D spans left column (both rows) ---
fig = plt.figure(figsize=(15, 9))
gs = fig.add_gridspec(2, 3, width_ratios=[1.6, 1, 1], wspace=0.25, hspace=0.3)

# ---- (a) 3D scatter (spans left column) ----
ax3d = fig.add_subplot(gs[:, 0], projection="3d")
sc = ax3d.scatter(Z[:, 0], Z[:, 1], Z[:, 2],
                  c=orientation_labels, cmap=cmap, norm=norm,
                  s=8, alpha=0.6)
ax3d.set_xlabel("z1"); ax3d.set_ylabel("z2"); ax3d.set_zlabel("z3")
ax3d.set_title(f"3D Latent — Poisson")
ax3d.view_init(elev=20, azim=35)

# ---- (b) three pairwise 2D projections ----
pairs = [(0, 1), (0, 2), (1, 2)]
proj_axes = [
    fig.add_subplot(gs[0, 1]),  # z1 vs z2
    fig.add_subplot(gs[0, 2]),  # z1 vs z3
    fig.add_subplot(gs[1, 1]),  # z2 vs z3
]
for ax, (i, j) in zip(proj_axes, pairs):
    ax.scatter(Z[:, i], Z[:, j], c=orientation_labels,
               cmap=cmap, norm=norm, s=8, alpha=0.6)
    ax.set_xlabel(f"z{i+1}"); ax.set_ylabel(f"z{j+1}")
    ax.set_title(f"z{i+1} vs z{j+1}")
    ax.set_aspect("equal", adjustable="datalim")

# ---- (c) shared colorbar in the bottom-right empty cell ----
cax = fig.add_subplot(gs[1, 2])
cbar = fig.colorbar(
    sc,
    cax=cax,
    boundaries=norm.boundaries,   # reuse the SAME edges as the scatter's BoundaryNorm
    ticks=orients,                # labels at bin centers (0, 45, ..., 315)
    spacing="proportional",       # draw swatches between boundaries, not tick marks
)
cbar.ax.set_yticklabels([f"{o}°" for o in orients])
cbar.set_label("orientation")

plt.suptitle(f"3D Latent Space — Poisson Likelihood", fontsize=14)
plt.show()